## Driver Drowsiness Detection — ResNet50V2 (Transfer Learning)
Same 6-class task as the EfficientNet-B0 notebook, but using ResNet50V2 pretrained on ImageNet.
Unlike EfficientNet, ResNet has no internal preprocessing — `preprocess_input` is applied explicitly.

In [ ]:
import numpy as np
import pandas as pd
import os
import cv2
import matplotlib.pyplot as plt

In [ ]:
labels = os.listdir("/kaggle/input/datasets/dheerajperumandla/drowsiness-dataset/train")
print(labels)

# IMG_SIZE = 224 for ResNet50V2

In [ ]:
IMG_SIZE = 224

# for yawn and no_yawn — detect and crop face region

In [ ]:
def face_for_yawn(direc="/kaggle/input/datasets/dheerajperumandla/drowsiness-dataset/train",
                  yunet_path="/kaggle/input/datasets/youssefelfeky12/yunet-face-detection/face_detection_yunet_2023mar.onnx"):
    yaw_no = []
    categories = ["yawn", "no_yawn"]
    # YuNet replaces haarcascade_frontalface_default.xml.
    # Initial input size is reset per-image via setInputSize().
    detector = cv2.FaceDetectorYN.create(
        yunet_path, "", (320, 320),
        score_threshold=0.6, nms_threshold=0.3, top_k=50,
    )
    for category in categories:
        path_link = os.path.join(direc, category)
        class_num1 = categories.index(category)
        print(class_num1)
        for image in os.listdir(path_link):
            image_array = cv2.imread(os.path.join(path_link, image), cv2.IMREAD_COLOR)
            if image_array is None:
                continue
            h, w = image_array.shape[:2]
            detector.setInputSize((w, h))
            _, faces = detector.detect(image_array)
            if faces is None:
                continue
            for f in faces:
                x, y, fw, fh = int(f[0]), int(f[1]), int(f[2]), int(f[3])
                # clamp — YuNet can return slightly out-of-bounds boxes
                # on extreme poses
                x0, y0 = max(0, x), max(0, y)
                x1, y1 = min(w, x + fw), min(h, y + fh)
                if x1 <= x0 or y1 <= y0:
                    continue
                roi_color = image_array[y0:y1, x0:x1]
                resized_array = cv2.resize(roi_color, (IMG_SIZE, IMG_SIZE))
                yaw_no.append([resized_array, class_num1])
    return yaw_no


yawn_no_yawn = face_for_yawn()

# for closed and open eye

In [ ]:
def get_data(dir_path="/kaggle/input/datasets/dheerajperumandla/drowsiness-dataset/train"):
    labels = ['Closed', 'Open']
    data = []
    for label in labels:
        path = os.path.join(dir_path, label)
        class_num = labels.index(label) + 2
        print(class_num)
        for img in os.listdir(path):
            try:
                img_array = cv2.imread(os.path.join(path, img), cv2.IMREAD_COLOR)
                resized_array = cv2.resize(img_array, (IMG_SIZE, IMG_SIZE))
                data.append([resized_array, class_num])
            except Exception as e:
                print(e)
    return data


data_train = get_data()

# for head pose (front and down)

In [ ]:
def get_head_pose_data(dir_path="/kaggle/input/datasets/antuchowdhury112/headpose/train"):
    categories = ["front", "down"]
    head_data = []
    for category in categories:
        path = os.path.join(dir_path, category)
        class_num = categories.index(category) + 4
        print(class_num)
        for img in os.listdir(path):
            try:
                img_array = cv2.imread(os.path.join(path, img), cv2.IMREAD_COLOR)
                if img_array is None:
                    continue
                resized_array = cv2.resize(img_array, (IMG_SIZE, IMG_SIZE))
                if resized_array.shape != (IMG_SIZE, IMG_SIZE, 3):
                    continue
                head_data.append([resized_array, class_num])
            except Exception as e:
                print(e)
    return head_data

# extend data and convert array

In [ ]:
def append_data():
    yaw_no = face_for_yawn()
    data = get_data()
    head_data = get_head_pose_data()
    yaw_no.extend(data)
    yaw_no.extend(head_data)
    features = np.array([item[0] for item in yaw_no])
    labels = np.array([item[1] for item in yaw_no])
    return list(zip(features, labels))

# new variable to store

In [ ]:
new_data = append_data()

# separate label and features

In [ ]:
X = []
y = []
for feature, label in new_data:
    X.append(feature)
    y.append(label)

# reshape the array

In [ ]:
X = np.array(X)
X = X.reshape(-1, IMG_SIZE, IMG_SIZE, 3)

# LabelBinarizer

In [ ]:
from sklearn.preprocessing import LabelBinarizer
label_bin = LabelBinarizer()
y = label_bin.fit_transform(y)

# label array

In [ ]:
y = np.array(y)

# train test split

In [ ]:
from sklearn.model_selection import train_test_split
seed = 42
test_size = 0.30
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=seed, test_size=test_size)

# length of X_test

In [ ]:
len(X_test)

# import dependencies

In [ ]:
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.applications import ResNet50V2
from tensorflow.keras.applications.resnet_v2 import preprocess_input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import tensorflow as tf

# Data Augmentation
ResNet50V2 has no internal normalization — `preprocess_input` scales pixels to [-1, 1].

In [ ]:
train_generator = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    zoom_range=0.2,
    horizontal_flip=True,
    rotation_range=30
)
test_generator = ImageDataGenerator(preprocessing_function=preprocess_input)

train_generator = train_generator.flow(np.array(X_train), y_train, batch_size=32, shuffle=False)
test_generator  = test_generator.flow(np.array(X_test),  y_test,  batch_size=32, shuffle=False)

# Model — ResNet50V2 (Transfer Learning)
## Phase 1: freeze the entire base model, train only the new head
A BatchNormalization layer is added after GAP — ResNet was designed with BN throughout.

In [ ]:
base_model = ResNet50V2(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)
base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = BatchNormalization()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.3)(x)
output = Dense(6, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=output)

model.compile(
    loss='categorical_crossentropy',
    metrics=['accuracy'],
    optimizer=Adam(learning_rate=1e-3)
)

model.summary()

# Phase 1 Training — head only

In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
reduce_lr  = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6)

history1 = model.fit(
    train_generator,
    epochs=20,
    validation_data=test_generator,
    validation_steps=len(test_generator),
    shuffle=True,
    callbacks=[early_stop, reduce_lr]
)

# Phase 2 — Fine-tuning
## Unfreeze the last 30 layers of ResNet50V2 and train with a very low learning rate

In [ ]:
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    loss='categorical_crossentropy',
    metrics=['accuracy'],
    optimizer=Adam(learning_rate=1e-5)
)

early_stop2 = EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True)
reduce_lr2  = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7)

history2 = model.fit(
    train_generator,
    epochs=40,
    validation_data=test_generator,
    validation_steps=len(test_generator),
    shuffle=True,
    callbacks=[early_stop2, reduce_lr2]
)

# Training history — combined Phase 1 + Phase 2

In [ ]:
acc      = history1.history['accuracy']     + history2.history['accuracy']
val_acc  = history1.history['val_accuracy'] + history2.history['val_accuracy']
loss     = history1.history['loss']         + history2.history['loss']
val_loss = history1.history['val_loss']     + history2.history['val_loss']
epochs   = range(len(acc))

plt.plot(epochs, acc,     'b', label='Training accuracy')
plt.plot(epochs, val_acc, 'r', label='Validation accuracy')
plt.axvline(x=len(history1.history['accuracy']), color='gray', linestyle='--', label='Fine-tune start')
plt.legend()
plt.title('Accuracy')
plt.show()

plt.plot(epochs, loss,     'b', label='Training loss')
plt.plot(epochs, val_loss, 'r', label='Validation loss')
plt.axvline(x=len(history1.history['loss']), color='gray', linestyle='--', label='Fine-tune start')
plt.legend()
plt.title('Loss')
plt.show()

# save model

In [ ]:
model.save('drowsiness_resnet50v2.h5')
model.save('drowsiness_resnet50v2.keras')

# Prediction

In [ ]:
# ResNet50V2 requires explicit preprocessing — no internal normalization layer
X_test_proc = preprocess_input(X_test.astype('float32'))
prediction  = np.argmax(model.predict(X_test_proc), axis=1)

# classification report

In [ ]:
labels_new = ['yawn', 'no_yawn', 'Closed', 'Open', 'front', 'down']

from sklearn.metrics import classification_report
print(classification_report(np.argmax(y_test, axis=1), prediction, target_names=labels_new))

---
# Extended Analysis

In [ ]:
import seaborn as sns
from sklearn.metrics import (
    confusion_matrix, roc_curve, auc,
    precision_recall_curve, average_precision_score
)
from sklearn.manifold import TSNE

labels_new  = ['yawn', 'no_yawn', 'Closed', 'Open', 'front', 'down']
true_labels = np.argmax(y_test, axis=1)
# ResNet50V2 has no internal preprocessing — must apply preprocess_input before inference
X_test_proc = preprocess_input(X_test.astype('float32'))
y_prob      = model.predict(X_test_proc)
predictions = np.argmax(y_prob, axis=1)

## 1. Class Distribution

In [ ]:
class_counts_train = np.bincount(np.argmax(y_train, axis=1), minlength=6)
class_counts_test  = np.bincount(true_labels, minlength=6)

x = np.arange(6)
w = 0.35
plt.figure(figsize=(9, 4))
plt.bar(x - w/2, class_counts_train, w, label='Train')
plt.bar(x + w/2, class_counts_test,  w, label='Test')
plt.xticks(x, labels_new)
plt.ylabel('Count')
plt.title('Class Distribution — Train vs Test')
plt.legend()
plt.tight_layout()
plt.show()

## 2. Confusion Matrix

In [ ]:
cm = confusion_matrix(true_labels, predictions)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels_new, yticklabels=labels_new)
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

## 3. Per-Class Accuracy

In [ ]:
per_class_acc = cm.diagonal() / cm.sum(axis=1)

plt.figure(figsize=(8, 4))
bars = plt.bar(labels_new, per_class_acc, color='steelblue')
plt.ylim(0, 1.15)
plt.title('Per-Class Accuracy')
plt.ylabel('Accuracy')
for bar, acc in zip(bars, per_class_acc):
    plt.text(bar.get_x() + bar.get_width() / 2, acc + 0.02,
             f'{acc:.1%}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

## 4. ROC Curves + AUC (One-vs-Rest)

In [ ]:
plt.figure(figsize=(9, 6))
for i, label in enumerate(labels_new):
    fpr, tpr, _ = roc_curve(y_test[:, i], y_prob[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f'{label} (AUC={roc_auc:.3f})')
plt.plot([0, 1], [0, 1], 'k--', linewidth=0.8)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves — One-vs-Rest')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

## 5. Precision-Recall Curves

In [ ]:
plt.figure(figsize=(9, 6))
for i, label in enumerate(labels_new):
    precision, recall, _ = precision_recall_curve(y_test[:, i], y_prob[:, i])
    ap = average_precision_score(y_test[:, i], y_prob[:, i])
    plt.plot(recall, precision, label=f'{label} (AP={ap:.3f})')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curves')
plt.legend(loc='upper right')
plt.tight_layout()
plt.show()

## 6. Misclassification Gallery

In [ ]:
wrong_idx = np.where(predictions != true_labels)[0]
n_show = min(16, len(wrong_idx))

fig, axes = plt.subplots(4, 4, figsize=(12, 12))
for i, ax in enumerate(axes.flat):
    if i >= n_show:
        ax.axis('off')
        continue
    idx = wrong_idx[i]
    img_rgb = cv2.cvtColor(X_test[idx].astype('uint8'), cv2.COLOR_BGR2RGB)
    ax.imshow(img_rgb)
    ax.set_title(
        f'True: {labels_new[true_labels[idx]]}\nPred: {labels_new[predictions[idx]]}',
        fontsize=8
    )
    ax.axis('off')
plt.suptitle(f'Misclassified Images ({len(wrong_idx)} total)', fontsize=13)
plt.tight_layout()
plt.show()

## 7. Grad-CAM — one sample per class

In [ ]:
def make_gradcam_heatmap(img_array, model, last_conv_layer_name):
    grad_model = tf.keras.models.Model(
        inputs=model.input,
        outputs=[model.get_layer(last_conv_layer_name).output, model.output]
    )
    with tf.GradientTape() as tape:
        conv_outputs, preds = grad_model(img_array)
        pred_index = tf.argmax(preds[0])
        class_score = preds[:, pred_index]
    grads = tape.gradient(class_score, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    heatmap = conv_outputs[0] @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()


def overlay_gradcam(img_bgr, heatmap, alpha=0.4):
    h, w = img_bgr.shape[:2]
    heatmap_resized = cv2.resize(heatmap, (w, h))
    heatmap_colored = cv2.applyColorMap(np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET)
    superimposed = cv2.addWeighted(img_bgr.astype('uint8'), 1 - alpha, heatmap_colored, alpha, 0)
    return cv2.cvtColor(superimposed, cv2.COLOR_BGR2RGB)


# dynamically find the last Conv2D layer (ResNet50V2 → last residual block conv)
last_conv_layer_name = next(
    layer.name for layer in reversed(model.layers)
    if isinstance(layer, tf.keras.layers.Conv2D)
)
print(f'Grad-CAM target layer: {last_conv_layer_name}')

fig, axes = plt.subplots(2, 6, figsize=(18, 7))
for i, label in enumerate(labels_new):
    idxs = np.where(true_labels == i)[0]
    idx  = idxs[0]
    img_bgr   = X_test[idx].astype('uint8')
    # ResNet50V2 needs preprocess_input — apply before feeding to Grad-CAM
    img_input = preprocess_input(np.expand_dims(img_bgr.astype('float32'), axis=0))
    heatmap   = make_gradcam_heatmap(img_input, model, last_conv_layer_name)
    overlay   = overlay_gradcam(img_bgr, heatmap)

    axes[0, i].imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    axes[0, i].set_title(f'{label}\n(original)', fontsize=8)
    axes[0, i].axis('off')

    axes[1, i].imshow(overlay)
    axes[1, i].set_title(f'{label}\n(Grad-CAM)', fontsize=8)
    axes[1, i].axis('off')

plt.suptitle('Grad-CAM Heatmaps — one sample per class', fontsize=13)
plt.tight_layout()
plt.show()

## 8. t-SNE of Feature Embeddings (GAP layer)

In [ ]:
# build a sub-model that outputs GlobalAveragePooling2D activations
gap_layer_name = next(
    layer.name for layer in model.layers
    if isinstance(layer, tf.keras.layers.GlobalAveragePooling2D)
)
feature_extractor = tf.keras.models.Model(
    inputs=model.input,
    outputs=model.get_layer(gap_layer_name).output
)

n_tsne     = min(2000, len(X_test))
idx_subset = np.random.choice(len(X_test), n_tsne, replace=False)
# ResNet50V2: apply preprocess_input before extracting features
embeddings = feature_extractor.predict(
    preprocess_input(X_test[idx_subset].astype('float32'))
)

tsne   = TSNE(n_components=2, random_state=42, perplexity=40, n_iter=1000)
emb_2d = tsne.fit_transform(embeddings)

plt.figure(figsize=(10, 8))
colors = plt.cm.tab10(np.linspace(0, 1, 6))
for i, label in enumerate(labels_new):
    mask = true_labels[idx_subset] == i
    plt.scatter(emb_2d[mask, 0], emb_2d[mask, 1],
                c=[colors[i]], label=label, alpha=0.6, s=15)
plt.legend(markerscale=2)
plt.title('t-SNE of Feature Embeddings (GAP layer)')
plt.xlabel('t-SNE dim 1')
plt.ylabel('t-SNE dim 2')
plt.tight_layout()
plt.show()

# predicting function

In [ ]:
labels_new = ['yawn', 'no_yawn', 'Closed', 'Open', 'front', 'down']

def prepare(filepath):
    img_array = cv2.imread(filepath, cv2.IMREAD_COLOR)
    resized   = cv2.resize(img_array, (IMG_SIZE, IMG_SIZE)).astype('float32')
    # ResNet50V2 requires explicit preprocessing
    resized   = preprocess_input(resized)
    return resized.reshape(-1, IMG_SIZE, IMG_SIZE, 3)

model = tf.keras.models.load_model('drowsiness_resnet50v2.h5')

# Prediction
## 0-yawn, 1-no_yawn, 2-Closed, 3-Open, 4-front, 5-down

In [ ]:
prediction = model.predict([prepare('/kaggle/input/datasets/dheerajperumandla/drowsiness-dataset/train/no_yawn/1028.jpg')])
np.argmax(prediction)

In [ ]:
prediction = model.predict([prepare('/kaggle/input/datasets/dheerajperumandla/drowsiness-dataset/train/Closed/_108.jpg')])
np.argmax(prediction)

In [ ]:
prediction = model.predict([prepare('/kaggle/input/datasets/dheerajperumandla/drowsiness-dataset/train/Open/_111.jpg')])
np.argmax(prediction)

In [ ]:
prediction = model.predict([prepare('/kaggle/input/datasets/dheerajperumandla/drowsiness-dataset/train/yawn/109.jpg')])
np.argmax(prediction)